In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df = spark.read.format('delta').load("abfss://878eac13-677f-422e-b9fa-bae10de1d354@onelake.dfs.fabric.microsoft.com/62582fb9-8cbf-4f28-aa83-e002f40371ec/Tables/raw/CRM_CUST_BASE_RAW")
df.show()

StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 3, Finished, Available, Finished, False)

+-----------+------------------+---------------+-------------+-----------+
|Cust_Ref_ID|         Full_Name| [Contact_Info]|Joined_Date_@|Gender_Code|
+-----------+------------------+---------------+-------------+-----------+
|      C-551|        ARUN KUMAR|       9.20E+11|   12-05-2024|          M|
|      C-882|      Priya_Sharma|  91-8877665544|   15-06-2024|          F|
|      C-102|      vikram singh|     7766554433| Jan 10, 2024|          M|
|      C-404|       Anjali Nair|+91 90001 10002|   2024.11.20|          F|
|      C-999|          Rahul .K|           NULL|   01-12-2024|          M|
|      C-551|        ARUN KUMAR|     9876543210|   12-05-2024|          M|
|      C-202|Sneha (Tech Store)|     9922334455|   01-01-2025|          F|
+-----------+------------------+---------------+-------------+-----------+



AttributeError: 'DataFrame' object has no attribute 'dtype'

In [2]:
df = df.selectExpr(
    "Cust_Ref_ID as cust_ref_id", 
    "Full_Name as full_name", 
    "`[Contact_Info]` as contact_info", 
    "`Joined_Date_@` as joined_date", 
    "Gender_Code as gender_code"
)
df.show()

StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 4, Finished, Available, Finished, False)

+-----------+------------------+---------------+------------+-----------+
|cust_ref_id|         full_name|   contact_info| joined_date|gender_code|
+-----------+------------------+---------------+------------+-----------+
|      C-551|        ARUN KUMAR|       9.20E+11|  12-05-2024|          M|
|      C-882|      Priya_Sharma|  91-8877665544|  15-06-2024|          F|
|      C-102|      vikram singh|     7766554433|Jan 10, 2024|          M|
|      C-404|       Anjali Nair|+91 90001 10002|  2024.11.20|          F|
|      C-999|          Rahul .K|           NULL|  01-12-2024|          M|
|      C-551|        ARUN KUMAR|     9876543210|  12-05-2024|          M|
|      C-202|Sneha (Tech Store)|     9922334455|  01-01-2025|          F|
+-----------+------------------+---------------+------------+-----------+



In [9]:
from pyspark.sql.functions import regexp_replace, col, initcap
from pyspark.sql import functions as F

df = df.withColumn("full_name", regexp_replace(col("full_name"), "_", " "))
df = df.withColumn("full_name", regexp_replace(col("full_name"), r"\(.*?\)", ""))
df = df.withColumn("full_name", F.initcap(F.col("full_name")))

df.show()

StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 11, Finished, Available, Finished, False)

+-----------+------------+------------+------------+-----------+
|cust_ref_id|   full_name|contact_info| joined_date|gender_code|
+-----------+------------+------------+------------+-----------+
|      C-551|  Arun Kumar|     unknown|  12-05-2024|          M|
|      C-882|Priya Sharma|  8877665544|  15-06-2024|          F|
|      C-102|Vikram Singh|  7766554433|Jan 10, 2024|          M|
|      C-404| Anjali Nair|  9000110002|  2024.11.20|          F|
|      C-999|    Rahul .k|     unknown|  01-12-2024|          M|
|      C-551|  Arun Kumar|  9876543210|  12-05-2024|          M|
|      C-202|      Sneha |  9922334455|  01-01-2025|          F|
+-----------+------------+------------+------------+-----------+



In [10]:
from pyspark.sql.functions import coalesce, lit
df = df.withColumn("contact_info", regexp_replace(col("contact_info"), "[^0-9]", ""))
df = df.withColumn("contact_info", 
    regexp_replace(col("contact_info"), "^91|^0", "")
)

df = df.withColumn("contact_info", col("contact_info").substr(-10, 10))
df = df.withColumn("contact_info",
    F.when((F.length("contact_info") != 10), "unknown")
    .otherwise(F.col("contact_info"))
)
df.show()

StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 12, Finished, Available, Finished, False)

+-----------+------------+------------+------------+-----------+
|cust_ref_id|   full_name|contact_info| joined_date|gender_code|
+-----------+------------+------------+------------+-----------+
|      C-551|  Arun Kumar|     unknown|  12-05-2024|          M|
|      C-882|Priya Sharma|  8877665544|  15-06-2024|          F|
|      C-102|Vikram Singh|  7766554433|Jan 10, 2024|          M|
|      C-404| Anjali Nair|  9000110002|  2024.11.20|          F|
|      C-999|    Rahul .k|     unknown|  01-12-2024|          M|
|      C-551|  Arun Kumar|  9876543210|  12-05-2024|          M|
|      C-202|      Sneha |  9922334455|  01-01-2025|          F|
+-----------+------------+------------+------------+-----------+



In [14]:
from pyspark.sql.functions import to_date, coalesce, col, date_format
df = df.withColumn(
   "joined_date",
   date_format(
       coalesce(
           to_date(col("joined_date"), "MMM dd, yyyy"),
           to_date(col("joined_date"), "yyyy.MM.dd"), 
           to_date(col("joined_date"), "dd-MM-yyyy")
       ),
       "yyyy-MM-dd"
   )
)
df.show()

StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 16, Finished, Available, Finished, False)

+-----------+------------+------------+-----------+-----------+
|cust_ref_id|   full_name|contact_info|joined_date|gender_code|
+-----------+------------+------------+-----------+-----------+
|      C-551|  Arun Kumar|     unknown| 2024-05-12|       Male|
|      C-882|Priya Sharma|  8877665544| 2024-06-15|     Female|
|      C-102|Vikram Singh|  7766554433| 2024-01-10|       Male|
|      C-404| Anjali Nair|  9000110002| 2024-11-20|     Female|
|      C-999|    Rahul .k|     unknown| 2024-12-01|       Male|
|      C-551|  Arun Kumar|  9876543210| 2024-05-12|       Male|
|      C-202|      Sneha |  9922334455| 2025-01-01|     Female|
+-----------+------------+------------+-----------+-----------+



In [15]:
from pyspark.sql import functions as F

df = df.withColumn(
    "gender_code", 
    F.when(F.col("gender_code") == "M", "Male") 
    .otherwise(F.col("gender_code")))
df = df.withColumn(
    "gender_code", 
    F.when(F.col("gender_code") == "F", "Female") 
    .otherwise(F.col("gender_code"))) 
df.show()


StatementMeta(, ca70dc88-5187-4d00-af9e-f0da6a369543, 17, Finished, Available, Finished, False)

+-----------+------------+------------+-----------+-----------+
|cust_ref_id|   full_name|contact_info|joined_date|gender_code|
+-----------+------------+------------+-----------+-----------+
|      C-551|  Arun Kumar|     unknown| 2024-05-12|       Male|
|      C-882|Priya Sharma|  8877665544| 2024-06-15|     Female|
|      C-102|Vikram Singh|  7766554433| 2024-01-10|       Male|
|      C-404| Anjali Nair|  9000110002| 2024-11-20|     Female|
|      C-999|    Rahul .k|     unknown| 2024-12-01|       Male|
|      C-551|  Arun Kumar|  9876543210| 2024-05-12|       Male|
|      C-202|      Sneha |  9922334455| 2025-01-01|     Female|
+-----------+------------+------------+-----------+-----------+

